In [ ]:
import pyarrow.dataset as ds
import gcsfs
import duckdb
import time
import os

# ADC 인증
GCP_PROJECT_ID = 'project-PROJECT-NAME'  # 프로젝트 이름
fs = gcsfs.GCSFileSystem(project=GCP_PROJECT_ID, token='google_default')

# 변환 전 GCS 원본 용량 계산
print("GCS 원본 용량 확인 중...")
gcs_info = fs.du('gdelt-eda-0331/', deep=True, total=False)
pre_total_bytes = sum(size for path, size in gcs_info.items() if path.endswith('.parquet'))
pre_total_mb = pre_total_bytes / (1024**2)
print(f"원본 데이터 용량: {pre_total_mb:.2f} MB")

# 데이터 스캔 및 데이터셋 정의
print("GCS 파일 스캔 중...")
all_files = fs.glob('gdelt-eda-0331/**/*.parquet')
raw_files = [f for f in all_files if 'final_data' not in f and 'fixed' not in f and 'final' not in f]

dataset = ds.dataset('gdelt-eda-0331/', format='parquet', filesystem=fs)
con = duckdb.connect()

# 관련 없는 국가 코드 삭제
remove_countries = "('SU', 'DJ', 'ET', 'ER', 'MZ', 'IN', 'GR', 'UZ', 'AM', 'AJ', 'BU', 'RS', 'OS', 'GG', 'TX')"

print("데이터 스캔 및 최종 가공 시작")
start_time = time.time()

# 메인 데이터 처리 (별칭 적용, 필터링, 분할 동시 수행)
main_query = f"""
COPY (
    SELECT
        GLOBALEVENTID::UINT32 AS GLOBALEVENTID,
        SQLDATE::INT32 AS SQLDATE,
        Actor1Name, Actor1CountryCode,
        Actor2Name, Actor2CountryCode,
        IsRootEvent::TINYINT AS IsRootEvent,
        TRY_CAST(EventCode AS INT16) AS EventCode,           -- '--' 2행 NULL 처리
        TRY_CAST(EventRootCode AS INT8) AS EventRootCode,    -- 동일하게 처리
        QuadClass::TINYINT AS QuadClass,
        GoldsteinScale::FLOAT AS GoldsteinScale,
        NumMentions::INTEGER AS NumMentions,
        NumArticles::INT16 AS NumArticles,       -- 추가
        NumSources::INT16 AS NumSources,         -- 추가
        AvgTone::FLOAT AS AvgTone,
        split_part(ActionGeo_FullName, ',', 1) AS ActionGeo_FullName,
        ActionGeo_CountryCode,
        ActionGeo_Lat::FLOAT AS ActionGeo_Lat,
        ActionGeo_Long::FLOAT AS ActionGeo_Long,
        ActionGeo_FeatureID,
        ActionGeo_Type::TINYINT AS ActionGeo_Type
    FROM dataset
    WHERE ActionGeo_CountryCode NOT IN {remove_countries}
) TO 'gdelt_main_0331.parquet' (FORMAT PARQUET, COMPRESSION 'ZSTD');
"""

print("1/2: gdelt_main_0331.parquet 생성 중...")
con.execute(main_query)

# URL 데이터 처리
# 'gdelt_main_0331.parquet'의 ID와 URL 매칭
url_query = """
COPY (
    SELECT 
        d.GLOBALEVENTID::UINT32 AS GLOBALEVENTID,
        d.SOURCEURL
    FROM dataset d
    SEMI JOIN 'gdelt_main_0331.parquet' m 
    ON d.GLOBALEVENTID = m.GLOBALEVENTID
) TO 'gdelt_url_0331.parquet' (FORMAT PARQUET, COMPRESSION 'ZSTD');
"""

print("gdelt_url_0331.parquet 생성 중...")
con.execute(url_query)
print(f"모든 작업 완료! 소요 시간: {(time.time() - start_time)/60:.2f}분")

# 변환 후 로컬 용량 계산 및 리포트 출력
post_main_bytes = os.path.getsize('gdelt_main_0331.parquet')
post_url_bytes = os.path.getsize('gdelt_url_0331.parquet')
post_total_mb = (post_main_bytes + post_url_bytes) / (1024**2)

print("-" * 30)
print("Data Optimization Report")
print("-" * 30)
print(f"GCS Original Size: {pre_total_mb:.2f} MB")
print(f"Local Final Size: {post_total_mb:.2f} MB")
print(f"Compression Ratio: {(post_total_mb / pre_total_mb) * 100:.2f}%")
print(f" - gdelt_main_0331: {post_main_bytes / (1024**2):.2f} MB")
print(f" - gdelt_url_0331: {post_url_bytes / (1024**2):.2f} MB")